# Исследовательский анализ рынка общественного питания Москвы

### Цели и задачи проекта

#### Цель 
- на основе предоставлнных данных предоставить заказчику рекомендации для открытия заведения общественного питания в Москве

#### Задачи
- ознакомиться с предоставленными данными 
- провести их предобработку
- провести исследовательский анализ, выявить закономерности
- исходя из выявленных законорностей, сформулировать рекомендации для заказчика 


### Описание данных

Предоставлен датасет с заведениями общественного питания Москвы, составленный на основе данных сервисов Яндекс Карты и Яндекс Бизнес на лето 2022 года. Информация, размещённая в сервисе Яндекс Бизнес, могла быть добавлена пользователями или найдена в общедоступных источниках. Она носит исключительно справочный характер.


Файл **/datasets/rest_info.csv** содержит информацию о заведениях общественного питания:
- name — название заведения;
- address — адрес заведения;
- district — административный район, в котором находится заведение, например Центральный административный округ;
- category — категория заведения, например «кафе», «пиццерия» или «кофейня»;
- hours — информация о днях и часах работы;
- rating — рейтинг заведения по оценкам пользователей в Яндекс Картах (высшая оценка — 5.0);
- chain — число, выраженное 0 или 1, которое показывает, является ли заведение сетевым (для маленьких сетей могут встречаться ошибки):
    - 0 — заведение не является сетевым;
    - 1 — заведение является сетевым.
- seats — количество посадочных мест.


Файл **/datasets/rest_price.csv** содержит информацию о среднем чеке в заведениях общественного питания:
- price — категория цен в заведении, например «средние», «ниже среднего», «выше среднего» и так далее;
- avg_bill — строка, которая хранит среднюю стоимость заказа в виде диапазона, например:
    «Средний счёт: 1000–1500 ₽»;
    «Цена чашки капучино: 130–220 ₽»;
    «Цена бокала пива: 400–600 ₽».
    и так далее;
- middle_avg_bill — число с оценкой среднего чека, которое указано только для значений из столбца avg_bill, начинающихся с подстроки «Средний счёт»:
    - Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
    - Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
    - Если значения нет или оно не начинается с подстроки «Средний счёт», то в столбец ничего не войдёт.
- middle_coffee_cup — число с оценкой одной чашки капучино, которое указано только для значений из столбца avg_bill, начинающихся с подстроки «Цена одной чашки капучино»:
    - Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
    - Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
    - Если значения нет или оно не начинается с подстроки «Цена одной чашки капучино», то в столбец ничего не войдёт.


### Содержимое проекта

1. Загрузка данных и знакомство с ними    
   
2. Предобработка данных

3. Исследовательский анализ данных

4. Итоговый вывод и рекомендации

---

## 1. Загрузка данных и знакомство с ними

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from phik import phik_matrix


In [ ]:
df_info = pd.read_csv('https://code.s3.yandex.net/datasets/rest_info.csv')

In [ ]:
df_info.head()

#### Анализ:

- датафрейм содержит 8406 строк, 9 столбцов
- пропуски есть в столбцых hours, seats
- В типах данных есть неточность для seats (может быть только натуральным числом), для корректности надо перевести от float к int
- после перевода seats к int можно понизить размерность
- chain можно перевести в bool, поскольку это признак (значения только 0, 1)

In [ ]:
# явные пропуски в данных
df_info.isna().sum() / len(df_info) * 100

In [ ]:
# проверка остальных столбцов
df_info['id'].nunique()  # все id уникальны

In [ ]:
# визуальная проверка на пропуски-флаги
df_info['category'].unique()

In [ ]:
df_info['district'].unique()

In [ ]:
df_info['rating'].unique()

In [ ]:
df_info['address'].unique()

In [ ]:
df_info['chain'].unique()

In [ ]:
df_info['seats'].unique()

на первый взгляд неявных пропусков нет

In [ ]:
# смотрим на дубликаты
df_info.duplicated().sum()
# явных дубликатов нет

In [ ]:
df_info.duplicated(subset='id').sum()  # все id уникальны, неявных дубликатов нет

#### Промежуточный итог для rest_info.csv:

- датафрейм содержит 8406 строк, 9 столбцов
- В типах данных есть неточность для seats (может быть только натуральным числом), для корректности надо перевести от float к int
- после перевода seats к int можно понизить размерность
- chain можно привести к bool (значения только 0, 1)
- явные пропуски: seats - 43% пропусков, hours - 6.4% пропусков. Пропусков слишком много, чтобы их удалять. Необходимо проводить анализ.
- на первый взгляд индикаторов-пропусков нет
- явных и неявных дубликатов нет

In [ ]:
df_price = pd.read_csv('https://code.s3.yandex.net/datasets/rest_price.csv')

In [ ]:
df_price.head()

In [ ]:
df_price.info()
# 4058 строк, 5 колонок
# пропуски в price, avg_bill, middle_avg_bill, middle_coffee_cup

In [ ]:
# явные пропуски
df_price.isna().sum() / df_price.shape[0] * 100

In [ ]:
# неявные пропуски
df_price['price'].unique()

In [ ]:
df_price['avg_bill'].unique()

In [ ]:
df_price['middle_avg_bill'].unique()

In [ ]:
df_price['middle_coffee_cup'].unique()

на первый взгляд неявных пропусков нет

In [ ]:
# явные Дубликаты 
df_price.duplicated().sum()

In [ ]:
df_price.duplicated(subset='id').sum()  # все id уникальны, неявных дубликатов нет

#### Промежуточный итог для rest_price.csv:

- датафрейм содержит 4058 строк, 5 колонок
- в типах данных ошибок нет
- пропуски в price: 18.3%, avg_bill: 5.96%, middle_avg_bill: 22.4%, middle_coffee_cup: 86.8%. Пропусков много, необходимо проводить анализ, с чем они связаны.
- визуально неявных пропусков нет (значений-флагов)
- явных и неявных дубликатов в данных нет


---

### Промежуточный вывод


Объем данных:
- rest_info содержит 8406 строк, 9 столбцов
- rest_price содержит 4058 строк, 5 столбцов

Соответствие описанию:
- данные соответствуют описанию

Пропуски:
- явные пропуски rest_info: seats - 43% пропусков, hours - 6.4% пропусков. Пропусков слишком много, чтобы их удалять. Необходимо проводить анализ.
- явные пропуски rest_price: price - 18.3%, avg_bill - 5.96%, middle_avg_bill - 22.4%, middle_coffee_cup - 86.8%. Пропусков много, необходимо проводить анализ, с чем они связаны.
- на первый взгляд индикаторов-пропусков нет

Дубликаты:
- явных и неявных дубликатов нет

Типы данных:
- для rest_price ошибок нет
- для rest_info необходимо внести коррективы по типам + оптимизировать размерность 
    - В типах данных есть неточность для seats (может быть только натуральным числом), для корректности надо перевести от float к int
    - после перевода seats к int можно понизить размерность
    - для столбца chain можно провести оптимизацию с понижением размерности (значения только 0, 1)

### Подготовка единого датафрейма

- Объедините данные двух датасетов в один, с которым вы и продолжите работу.

In [ ]:
df = df_info.merge(df_price, on='id', how='left') # объединяем по id 

## 2. Предобработка данных

Подготовьте данные к исследовательскому анализу:

- Изучите корректность типов данных и при необходимости проведите их преобразование.

In [ ]:
df.head()

In [ ]:
df.info()

Преобразование типов:
- chain переводим к bool (по смыслу это признак сетевой/не сетевой, значения [0, 1])
- seats (кол-во посадочных мест) - переведем в float с понижением размерности данных 
- rating от 0 до 5.0 - можно понизить размерность 
- для middle_avg_bill и middle_coffee_cup также можно понизить размерность

In [ ]:
df['chain'] = df['chain'].astype(bool)
df['chain']

In [ ]:
def float_downcast(col_name: str):
    return pd.to_numeric(df[col_name], downcast='float')

In [ ]:
col_to_float_downcast = ['rating', 'middle_avg_bill', 'middle_coffee_cup', 'seats']

for col in col_to_float_downcast:
    df[col] = float_downcast(col)

In [ ]:
df.info()

### Изучение и обработка пропусков

In [ ]:
df.isna().sum() / df.shape[0] * 100

Анализ:

- seats, hours, price, avg_bill, middle_coffee_cup, middle_avg_bill - пропусков существенное количество. Удалять не будем. Оставим их как есть
- при обработке никакие данные не удалялись, она прошла без потерь


### Обработка дубликатов

In [ ]:
df.info()

In [ ]:
df.duplicated().sum() # явных дублей нет 

In [ ]:
df['id'].nunique() # неявных дублей по id нет

In [ ]:
def to_snake_case(value):
    """Преобразует одно значение в snake_case."""
    if pd.isna(value):
        return value
    s = str(value).strip().lower()
    s = re.sub(r"[^\w\s-]", "", s, flags=re.UNICODE)  
    s = re.sub(r"[\s\-]+", "_", s)                  
    s = re.sub(r"_+", "_", s).strip("_") 
    return s

def normalize_columns_snake_case(df, columns):
    """Нормализует указанные столбцы DataFrame в snake_case."""
    df = df.copy()
    for col in columns:
        df[col] = df[col].apply(to_snake_case)
    return df

In [ ]:
df = normalize_columns_snake_case(df, ["name", "address", 'district'])


In [ ]:
df.head()

In [ ]:
# копия данных
tmp = df.copy()

In [ ]:
len(df[df.duplicated(subset=['name', 'address'], keep="first")]) / len(df) # есть дубли по адресу и имени


In [ ]:
# дублей мало: менее 0.06%
# Можно спокойно удалить
df = df[~df.duplicated(subset=['name', 'address'], keep="first")]

In [ ]:
print('Потеря при обработке дублей: ', (len(tmp) - len(df)) / len(tmp) * 100, '%')

- Для дальнейшей работы создайте столбец `is_24_7` с обозначением того, что заведение работает ежедневно и круглосуточно, то есть 24/7:
  - логическое значение `True` — если заведение работает ежедневно и круглосуточно;
  - логическое значение `False` — в противоположном случае.

In [ ]:
df['hours'].unique()

In [ ]:
df['is_24_7'] = df['hours'].eq('ежедневно, круглосуточно')

In [ ]:
# проверка на корректность разбиения
df['is_24_7'].value_counts()

In [ ]:
len(df[df['hours'] == 'ежедневно, круглосуточно'])

In [ ]:
len(df[~(df['hours'] == 'ежедневно, круглосуточно')])

разбиение корректно

---

### Промежуточный вывод


Исходное количество данных:
- Потерь при обработке пропусков нет
- проведена нормализация строковых данных для столбцов name, address, district
- проведен анализ дубликатов: явных дублей не выявлено, есть неявные дубли по name и address
- потеря при обработке неявных дублей менее 0.06%
- добавлен столбец is_24_7 с признаком круглосуточное заведение или нет

## 3. Исследовательский анализ данных

---

### Исследование количества объектов общественного питания по каждой категории.

In [ ]:
df.head()

In [ ]:
share = df['category'].value_counts(normalize=True).sort_values()



ax = share.plot(
    kind='barh', figsize=(10, 6), color='cornflowerblue'
)

ax.set_ylabel('Категория')          
ax.set_xlabel('Проценты')   
ax.set_title('Распределение заведений по категориям в ЦАО')

ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

for i, v in enumerate(share.values):
    ax.text(v + 0.003, i, f'{v:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Анализ:
- в данных преимущественно преставлены кафе: 28.3%, рестораны: 24.3%
- кофеен 16.8%
- баров,пабов и пиццерий и быстрого питания  7-9% по каждой
- самые малочисленные группы: столовые, булочные - менее 4% по каждой

---

### Исследование распределения количества заведений по административным районам Москвы.

In [ ]:
# Распределение по районам
district_share = df['district'].value_counts(normalize=True).sort_values()

ax = district_share.plot(
    kind='barh', figsize=(10, 6), color='cornflowerblue'
)

ax.set_ylabel('Район')          
ax.set_xlabel('Проценты')   
ax.set_title('Распределение заведений по районам')

ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

for i, v in enumerate(district_share.values):
    ax.text(v + 0.003, i, f'{v:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Анализ:
- заведений из центрального округа около 26.7%, что существенно больше, чем из остальных округов 
- в для 78% округов доля составляет от 8% до 11% 
- меньше всего заведений в северо-западном округе - 4.9%
- такое распределение по количеству заведений говорит о том, что в центре выше спрос на заведения общественного питания. Как следствие, там больше и предложение, больше потенциальных конкурентов.

In [ ]:
# Распределение в ЦАО по заведениям
central_district_data = df[df['district'] == 'центральный_административный_округ']
district_share = central_district_data['category'].value_counts(normalize=True).sort_values()
ax = district_share.plot(
    kind='barh', figsize=(10, 6), color='cornflowerblue'
)

ax.set_ylabel('Категория')          
ax.set_xlabel('Проценты')   
ax.set_title('Распределение заведений по категориям в ЦАО')

ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

for i, v in enumerate(district_share.values):
    ax.text(v + 0.003, i, f'{v:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Анализ:
- количество ресторанов лидирует с большим отрывом: около 30%
- по сравнению с данными по всем районам, есть небольшие отличия в распределении: 
    - доля ресторанов тут больше, чем кафе 
    - доля баров, пабов выше - 16.2% по сравнению с общими данными - 9.1%;
    - можно выделить малочисленные категории (пиццерии, быстрое питание, столовые и булочные - менее 5% каждая категория)

---

### Изучение соотношения сетевых и несетевых заведений 

In [ ]:
# изучение сетевых/несетевых заведений
chain_share = (
    central_district_data['chain']
    .value_counts(normalize=True)
    .rename({True: 'Сетевое', False: 'Не сетевое'})
)

ax = chain_share.plot(
    kind='pie',
    figsize=(6, 6),
    autopct='%1.1f%%', 
    startangle=90,
    colors=['cornflowerblue', 'lightgray']
)

ax.set_ylabel('')
ax.set_title('Доля сетевых и несетевых заведений')
plt.tight_layout()
plt.show()

### Анализ:
- в целом количество не сетевых 61% значительно больше сетевых 39%

In [ ]:
# соотношение сетевых/не сетевых в разрезе каждой группы
df_chain_unstack = df.groupby('category')['chain'].value_counts(normalize=True).unstack(fill_value=0)

df_chain_unstack.plot(kind='bar')


plt.title('Количество сетевых и не сетевых заведений по категориям') 
plt.ylabel('Количество заведений') 
plt.xlabel('Категория заведения') 
plt.xticks(rotation=45)  
plt.legend()  

# Отображаем график
plt.show()

In [ ]:
df.groupby('category')['chain'].value_counts(normalize=True).unstack(fill_value=0) * 100

### Анализ:
- **сетевые** заведения чаще встречаются в категориях:
    - булочная: 61%
    - кофейня: 50.9%
    - пиццерия: 52.1%
- в остальных категориях чаще встречаются **не сетевые**:
    - бар,паб:	78%
    - быстрое питание:	61%
    - кафе:	67%
    - ресторан:	64%
    - столовая:	72%

---

### Исследование количества посадочных мест в заведениях. 

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True)

sns.histplot(data=df, x='seats', ax=axes[0])
axes[0].set_title('Распределение seats')
axes[0].set_xlabel('')

sns.boxplot(data=df, x='seats', ax=axes[1])
axes[1].set_xlabel('seats')

plt.tight_layout()
plt.show()

In [ ]:
quantile = 0.95
extra_seats_bound = df['seats'].quantile(quantile)
print(f'{quantile * 100} % заведений имеют менее {extra_seats_bound} посадочных мест')

In [ ]:
df['seats'].describe()

### Анализ:
- медианное значение количества мест в заведениях: 75
- данные содержат выбросы. 
- имеются малочисленные заведения (менее 5% от общего числа) с количеством мест более 307 и до 1288

In [ ]:
# смотрим на распределения по местам в разрезе категорий
sns.boxplot(data=df, y='seats', x='category')
plt.xticks(rotation=45) 
plt.title('Распределение количества мест в зависимости от категории заведения')
plt.ylabel('распределение количества мест')
plt.show()

In [ ]:
# выбросы по каждой категории 
p90_by_cat = (
    df
    .groupby('category')['seats']
    .quantile(0.90)
    .rename('p90')
)

df_out = df.merge(p90_by_cat, on='category', how='left')
df_out['is_outlier_90'] = df_out['seats'] > df_out['p90']


outliers = df_out[df_out['is_outlier_90']].copy()
no_outliers = df_out[~df_out['is_outlier_90']].copy()


In [ ]:
# наиболее часто встречающиеся выбросы по категориям
most_freq_with_count = (
    outliers.groupby('category')['seats']
    .apply(lambda s: s.value_counts().head(1))
    .rename('count')
    .reset_index()
    .rename(columns={'level_1': 'seats'})
)

most_freq_with_count

### Анализ:
По распределению выбросов в каждой категории можно сделать предположения о их природе:
- выбросы могут быть связвано с наличием банкетных помещений, открытых фудкортов 
- наиболее часто встречающиеся выбросы в каждой категории имеют значения 250, 300, 320, 350 - это похоже на стандартную максимальную вместимость, которую указывают для заведений в открытых источниках. Скорее всего, это не реальное количество посадочных мест, а стандартное значение количества людей, которые гипотетически можно в это заведение привести и всем хватит места (не обязательно сидя).


In [ ]:
# наиболее типичное количество посадочных мест по категориям (отсеяли выбросы)
no_outliers.groupby('category')['seats'].median()

---

### Исследование рейтинга заведений.

In [ ]:
df['rating'].describe()

In [ ]:
avg_rating = df.groupby('category')['rating'].mean().sort_values(ascending=False)

In [ ]:
# средний рейтинг по каждой категории
avg_rating

In [ ]:
ax = avg_rating.plot(kind='bar', figsize=(10, 5), color='steelblue')
ax.set_xlabel('category')
ax.set_ylabel('Средний рейтинг')
ax.set_title('Средний рейтинг по категориям')
ax.grid(axis='y', linestyle='--', alpha=0.6)  # горизонтальные линии сетки
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print('Относительный размах по среднему рейтингу (%):')
(avg_rating.max() - avg_rating.min()) / avg_rating.max() * 100

### Анализ
- средний рейтинг мало отличается от вида заведения общепита. Размах по значениям 7.6%

---

### Корреляционный анализ. Связь с рейтингом заведения.

In [ ]:
df.head()

In [ ]:
# какие колонки будем изучать в рамках корреляционного анализа
numeric_columns = [
    'rating',
    'seats',
    'middle_avg_bill',
    'middle_coffee_cup'
]

categorial_columns = ['chain', 'district', 'is_24_7', 'category', 'price']

In [ ]:
df_rating_corr_analysis = df[numeric_columns + categorial_columns]

In [ ]:
# Посчитаем матрицу корреляций
corr_matrix = df_rating_corr_analysis.phik_matrix(interval_cols=numeric_columns)

# Создаём тепловую  карту
sns.heatmap(data=corr_matrix, annot=True, fmt='.2f', linewidths=0.5, cmap='viridis')

# Показываем график с заголовком
plt.title('Тепловая карта матрицы корреляций')
plt.show()
print("Корреляция переменных с 'rating':")
print(corr_matrix['rating'].sort_values( ascending=False))

Самая сильная положительная связь рейтинга с :
- категорией цены: корреляция 0.26

In [ ]:
sns.histplot(data=df_rating_corr_analysis, x='rating', hue='price', kde=True)
plt.grid()

plt.show() 

In [ ]:
avg_rating = df_rating_corr_analysis.groupby('price')['rating'].mean().sort_values(ascending=True)

ax = avg_rating.plot(kind='bar', figsize=(10, 5), color='steelblue')
ax.set_xlabel('category')
ax.set_ylabel('Средний рейтинг')
ax.set_title('Средний рейтинг по ценовым категориям')
ax.grid(axis='y', linestyle='--', alpha=0.6)  # горизонтальные линии сетки
plt.tight_layout()
plt.show()

In [ ]:
avg_rating

In [ ]:
print('Размах в процентах: ')
(avg_rating.max() - avg_rating.min()) / avg_rating.max() * 100

### Анализ:
- с увеличением ценового сегмента средний рейтинг имеет тенденцию расти. Но эта связь незначительная: коэффициент корреляции 0.26
- разброс среднего рейтинга по ценовым сегментам: около 5.93%

---

### Исследование самых популярных заведений Москвы


In [ ]:
top15_chains = (
    df[df['chain']]
    .groupby('name', as_index=False)
    .agg(
        category=('category','first'),
        establishments=('id', 'count'),
        mean_rating=('rating', 'mean')
    )
    .sort_values('establishments', ascending=False)
    .head(15)
)

top15_chains


In [ ]:
top15_chains_show = top15_chains.groupby('category')['establishments'].sum().sort_values() / top15_chains['establishments'].sum()
ax = top15_chains_show.plot(kind='bar', figsize=(10, 5), color='steelblue')
ax.grid(axis='y', linestyle='--', alpha=0.6)  # горизонтальные линии сетки
plt.xticks(rotation=45)
plt.title('Доля от общего количества заведений в топ 15 по категориям')
plt.tight_layout()
plt.show()

In [ ]:
top15_chains_show = top15_chains['category'].value_counts(normalize=True).sort_values()
ax = top15_chains_show.plot(kind='bar', figsize=(10, 5), color='steelblue')
ax.grid(axis='y', linestyle='--', alpha=0.6)  # горизонтальные линии сетки
plt.xticks(rotation=45)
plt.title('Доля категории по присутствию в рейтинге топ 15')
plt.tight_layout()
plt.show()

In [ ]:
top15_chains_show = top15_chains.groupby('category')['mean_rating'].mean().sort_values()
top15_chains_show

### Анализ:
Особенности самых популярных сетей:
- самое большое количество заведений среди кофеен
- присутствие в топ 15 больше всего у кофеен
- средний рейтинг среди популярных заведний высокий: более 4 
- в среднем рейтинг у популярных сетевых булочных выше, чем у других видов заведений. 
- разброс о среднему рейтингу среди популярных категорий сетей небольшой: 4.0-4.4 баллов (10%)

---

### Исследование среднего чека заведений


In [ ]:
# посмотрим на распеределение по районам
sns.boxplot(data=df, y='middle_avg_bill', x='district')
plt.xticks(rotation=45) 
plt.title('Распределение среднего чека по районам')
plt.show()

Имеются выбросы. В качестве статистики выберем медиану, чтобы избежать искажений от выбросов

In [ ]:
df.groupby('district')['middle_avg_bill'].median()


In [ ]:

def plot_heat_map_geo_grid(value: pd.Series, value_name: str, fmt: str='.0f'):
    geo_grid = pd.DataFrame(
        index=['Север', 'Центр', 'Юг'],
        columns=['Запад', 'Центр', 'Восток'],
        data=np.nan
    )

    geo_grid.loc['Север', 'Запад']   = value.get('северо_западный_административный_округ')
    geo_grid.loc['Север', 'Центр'] = value.get('северный_административный_округ')
    geo_grid.loc['Север', 'Восток']  = value.get('северо_восточный_административный_округ')

    geo_grid.loc['Центр', 'Запад']   = value.get('западный_административный_округ')
    geo_grid.loc['Центр', 'Центр'] = value.get('центральный_административный_округ')
    geo_grid.loc['Центр', 'Восток']  = value.get('восточный_административный_округ')

    geo_grid.loc['Юг', 'Запад']      = value.get('юго_западный_административный_округ')
    geo_grid.loc['Юг', 'Центр']    = value.get('южный_административный_округ')
    geo_grid.loc['Юг', 'Восток']     = value.get('юго_восточный_административный_округ')

    plt.figure(figsize=(8, 5))
    sns.heatmap(
        geo_grid,
        annot=True,
        fmt=fmt,
        cmap='YlOrRd',
        linewidths=1,
        linecolor='white',
        cbar_kws={'label': value_name}
    )
    plt.title('Тепловая карта по географии округов Москвы')
    plt.xlabel('')
    plt.ylabel('')
    plt.tight_layout()
    plt.show()


In [ ]:
# медианный средний чек по округам
bill = df.groupby('district')['middle_avg_bill'].median()
plot_heat_map_geo_grid(value=bill, value_name='Медианный средний чек')

In [ ]:
count_in_district = df['district'].value_counts(normalize=True)

plot_heat_map_geo_grid(value=count_in_district, value_name='Доля заведений', fmt='.3f')

---


### Анализ:
- в центре медианный чек выше, чем на окраине 
- также можно отметить, что на западе и северо-западе  более высокий чек, нежели на востоке и юго востоке
- центр крайне насыщен заведениями (не менее чем в 3 раза большепо сравнению любом другим округом)
- меньше всего конкурентов на северо-западе и юго-востоке 

---

### Промежуточный вывод

- В центральном районе точек общественного питания больше всего: около 33.5%. При этом в других районах точек общепита менее 11%.

- по категориям заведений лидируют кафе (28%) и рестораны (24.3%), далее идут кофейни (16.8%). К малочисленным сегментам можно отнести бары/пабы, быстрое питание, столовые и булочные (менее 9% у каждой категории)

- рынок преимущественно не сетевой (61%). При этом среди пиццерий, кофеен, булочных сетевых точек больше.

- медианное значение вместительности заведения около 78. В данных по вместительности присутствуют выбросы по каждой категории. Самые встречающиеся значение выбросов около 300 - технические, стандартные цифры по гипотетической вместимости заведения, а не реальное количество мест в заведении. Выбросы на 500+ можно объяснить фудкортами/банкетными залами.

- у рейтинга имеется слабая положительная связь с ценовым сегментом. Уровень корреляции 0.26; То есть с повышением ценового сегмента рейтинг имеет тенденцию расти, но несущественно.

- медианный чек высокий в центре. На западе/севоро-западе выше чем на востоке и юго-востоке

- в топовых по популярности сетях стабильно высокий рейтинг: медианное значение в каждой категории от 4.0 до 4.4

- среди популярных сети: кофейни




## 4. Итоговый вывод и рекомендации

1. В настоящей работе проведен анализ данных о заведениях общественного питания в городе Москве. Проделанная работа состояла из нескольких этапов:
    1) Выгрузка данных, проверка на соответствие заявленному описанию
    2) Предварительная обработка и приведение к формату, удобному для исследований.
    3) Проведение исследовательского анализа

2. По результатам ознакомления с данными и их предварительной обработки расхождений с заявленным описанием не выявлено, данные приведены к удобному для анализа формату с минимальными потерями (менее 0.06%). По результатам последующего анализа данных можно сформулировать ключевые выводы:
    - больше всего точек общественного питания в центре: 33.5%
    - топ 3 категории: кафе: 28.3%, рестораны: 24.3%, кофейни 16.8%
    - рынок преимущественно не сетевой: 61%. Однако для пиццерий, кофеен, булочных сетевых точек больше
    - характерное значение по вместимости заведений: медиана - 78 мест
    - выбросы по значениям вместимости (300+ мест) - формальные официальные цифры открытых источников, не отражающие реальное количество посадочных мест в заведениях
    - рейтинг слабо зависит от ценового сегмента. Корреляция по с этим признаком на уровне 0.26. 
    - в центральном районе медианный чек выше, чем на окраинах. Медианный чек на западе и северо-западе выше, чем на востоке и юго-востоке Москвы
    - самыми популярными по количеству точек являются кофейни 
    - рейтинг популярных заведений стабильно высокий: медианное знаяение оценок среди популярных сетей 4.0+

3. Рекомендации
    - Для обеспечения более высокого чека заведению стоит рассмотреть открытие в центре, на западе/северо-западе. 
    - При этом конкуренция в центре крайне высокая по сравнению с другими районами. Чтобы избежать большой конкуренции рекомендуется рассмотреть северо-западый и юго-восточный районы города. 
    - По формату заведения лучше избегать самых популярных ниш в выбранном районе, чтобы уменьшить влияние конкурентов. Стоит избегать открытия ресторанов и кафе в центре, поскольку рынок перенасыщен ими.
    - по вместимости заведения стоит ориентироваться на медианное значение в категории
    - рейтинг заведений не связан с его категорией, ценовым сегментом или районом. 
    - для запуска пиццерий, кофеен, булочных лучше выбрать сетевой формат, поскольку они статистически лучше приживаются на рынке. В остальных случаях можно смело выбирать не сетевой формат.
